In [ ]:
# from pydub import AudioSegment

# # merge audio
# audio = AudioSegment.empty()

# for i in range(1, 11):
#     path = f"../data/audio/poem1_versuri/vers{i}.wav"
#     audio += AudioSegment.from_wav(path)

# # Audio(audio.raw_data, rate=audio.frame_rate)
# audio

In [ ]:
from pydub import AudioSegment

audio = AudioSegment.from_file("../data/raw-audio/1. Sailing_to_byzantium_simona_final.wav")

# remove the intro
audio = audio[3_000:]

audio


## stress marker

In [ ]:
from dataclasses import fields
from typing import Any

from adapt_eeg.utils.stress_identifier import (
    StressIdentifyParams,
    detect_stressed_syllables,
)
from adapt_eeg.utils.syllable_identifier import SyllableIdentifyParams


class AllParams(StressIdentifyParams, SyllableIdentifyParams):

    def __init__(self, **kwargs) -> None:
        stress_fields = {f.name for f in fields(StressIdentifyParams)}
        syllable_fields = {f.name for f in fields(SyllableIdentifyParams)}

        StressIdentifyParams.__init__(
            self,
            **{k: v for k, v in kwargs.items() if k in stress_fields},
        )

        SyllableIdentifyParams.__init__(
            self,
            **{k: v for k, v in kwargs.items() if k in syllable_fields},
        )


params = AllParams(
    intensity_floor_hz=50.0,
    silence_threshold_relative_db=-25.0,
    min_nucleus_prominence_db=2.0,
    pitch_range_hz=(75.0, 500.0),
    pitch_time_step_s=0.01,
    voicing_half_window_s=0.04,
    min_voiced_fraction=0.50,
    min_pause_duration_s=0.2,
    pitch_percentile=90.0,
    #### syllables -> stress
    prominence_neighbor_half_window_s=0.05,
    prominence_min_nucleus_distance_s=0.04,
    min_pitch_prominence_hz=2.0,
    min_intensity_prominence_db=5.0,
    duration_neighbor_radius=3,
    stress_weights=(0.5, 0.3, 0.2),
    stress_score_threshold=-0.28,
)


#### syllables

In [ ]:
from adapt_eeg.utils.syllable_identifier import (
    detect_nucleus_candidates,
    get_f0_contour,
    get_intensity_contour,
    get_silence_threshold_db,
    load_audio,
)


sound, audio, samples, sample_rate = load_audio(audio)

intensity = get_intensity_contour(
    sound,
    params.intensity_floor_hz,
)

f0 = get_f0_contour(
    sound,
    pitch_time_step_s=params.pitch_time_step_s,
    pitch_range_hz=params.pitch_range_hz,
)

silence_threshold_db = get_silence_threshold_db(
    intensity,
    params.silence_threshold_relative_db,
)

nucleus_candidates = detect_nucleus_candidates(
    intensity,
    silence_threshold_db,
    params.min_nucleus_prominence_db,
)
nucleus_candidates_times = [x.time for x in nucleus_candidates]

In [ ]:
from adapt_eeg.utils.syllable_identifier import filter_voiced_nuclei


nuclei = filter_voiced_nuclei(
    nucleus_candidates,
    f0,
    params.voicing_half_window_s,
    params.min_voiced_fraction,
)

nuclei_times = [x.time for x in nuclei]

In [ ]:
from adapt_eeg.utils.syllable_identifier import estimate_syllable_intervals, measure_syllables


intervals = estimate_syllable_intervals(
    nuclei,
    intensity,
    sound.duration,
    params.min_pause_duration_s,
    silence_threshold_db,
)

syllables = measure_syllables(
    intervals,
    audio,
    samples,
    sample_rate,
    f0,
    params.pitch_percentile,
)

syllable_times = [x.nucleus for x in syllables]
len(syllables)

In [ ]:
from adapt_eeg.utils.syllable_identifier import Syllable


def find_syllable_idx_at_nucleus(
    syllables: list[Syllable],
    x: float,
) -> int | None:
    target = round(x, 2)

    left = 0
    right = len(syllables) - 1

    while left <= right:
        mid = (left + right) // 2
        nucleus = round(syllables[mid].nucleus, 2)

        if nucleus == target:
            return mid

        if nucleus < target:
            left = mid + 1
        else:
            right = mid - 1

    return None

#### Inspect long syllables

In [ ]:
from matplotlib import pyplot as plt


plt.hist([syllable.duration_s for syllable in syllables if syllable.duration_s >= 0.6])
plt.show()

In [ ]:
long_syllables = [syllable for syllable in syllables if syllable.duration_s >= 0.6]
long_syllables[2]._audio  # monument

In [ ]:
long_syllables[2].start

In [ ]:
long_syllables[4]._audio  # sing outloud 

In [ ]:
long_syllables[4].start

In [ ]:
long_syllables[6]._audio  ## too low volumn, ignore

In [ ]:
long_syllables[7]._audio  ## too low volumn, ignore

In [ ]:
long_syllables[8]._audio  ## just sing

#### stress

In [ ]:
from adapt_eeg.utils.stress_identifier import (
    _normalize_weights,
    calculate_stress_evidence,
    calculate_stress_scores,
    classify_stresses,
    normalize_stress_evidence,
)
from adapt_eeg.utils.syllable_identifier import SyllablizedAudio

# f0_stress = get_f0_contour(
#     sound, params.pitch_time_step_s, (165, 255)
# )

syllablized_audio = SyllablizedAudio(
    sound=sound,
    audio=audio,
    samples=samples,
    sample_rate=sample_rate,
    intensity=intensity,
    f0=f0,
    silence_threshold_db=silence_threshold_db,
    syllables=syllables,
)

evidence = calculate_stress_evidence(
    syllablized_audio,
    params.prominence_neighbor_half_window_s,
    params.prominence_min_nucleus_distance_s,
    duration_neighbor_radius=params.duration_neighbor_radius,
    pitch_floor_hz=params.pitch_range_hz[0],
)

normalized_evidence = normalize_stress_evidence(evidence)

weights = _normalize_weights(params.stress_weights)

scores = calculate_stress_scores(
    normalized_evidence,
    weights,
)

stress_flags = classify_stresses(
    evidence,
    scores,
    params.min_pitch_prominence_hz,
    params.min_intensity_prominence_db,
    params.stress_score_threshold,
)

stresses = [syllable for syllable, stressed in zip(syllables, stress_flags) if stressed]
stress_times = [stress.nucleus for stress in stresses]

In [ ]:
for i, (syllable_time, score, flag) in enumerate(zip(evidence, scores, stress_flags)):
    print(syllable_time, score, flag)
    if i > 10:
        break

In [ ]:
for i, (syllable_time, score, flag) in enumerate(zip(normalized_evidence, scores, stress_flags)):
    print(syllable_time, score, flag)
    if i > 10:
        break

In [ ]:
syllables[1], scores[1]

In [ ]:
# prominences[0]

In [ ]:
# prominences[1]

## visualize code



In [ ]:
import base64
import io
import json
import mimetypes
import uuid
from pathlib import Path

from IPython.display import HTML, display
from pydub import AudioSegment


def _audio_to_data_url(audio):
    if isinstance(audio, AudioSegment):
        buf = io.BytesIO()
        audio.export(buf, format="wav")
        audio_bytes = buf.getvalue()
        mime_type = "audio/wav"

    elif isinstance(audio, (str, Path)):
        path = Path(audio)

        if not path.exists():
            raise FileNotFoundError(path)

        audio_bytes = path.read_bytes()
        mime_type = mimetypes.guess_type(path.name)[0] or "audio/mpeg"

    else:
        raise TypeError("audio must be a pydub.AudioSegment or a file path")

    encoded = base64.b64encode(audio_bytes).decode("ascii")
    return f"data:{mime_type};base64,{encoded}"


def waveform_player(
    audio,
    timestamps,
    window_seconds=20,
    height=160,
):
    """
    Parameters
    ----------
    audio:
        pydub.AudioSegment or path to file

    timestamps:
        [1.2, 3.4, 8.1]

        or

        [
            (1.2, "marker A"),
            (3.4, "marker B"),
        ]

    window_seconds:
        Total visible time.
        20 means roughly +/- 10 seconds around playback.
    """

    audio_url = _audio_to_data_url(audio)

    markers = []

    for item in timestamps:
        if isinstance(item, (tuple, list)):
            t, label = item
        else:
            t = item
            label = f"{float(t):.2f}s"

        markers.append(
            {
                "time": float(t),
                "label": str(label),
            }
        )

    markers_json = json.dumps(markers)

    uid = uuid.uuid4().hex

    waveform_id = f"waveform_{uid}"
    button_id = f"button_{uid}"
    time_id = f"time_{uid}"

    html = f"""
    <style>
        #{waveform_id} {{
            width: 100%;
        }}

        .controls-{uid} {{
            margin-top: 10px;
            display: flex;
            align-items: center;
            gap: 12px;
            font-family: sans-serif;
        }}

        #{time_id} {{
            font-family: monospace;
            font-size: 13px;
        }}
    </style>

    <div id="{waveform_id}"></div>

    <div class="controls-{uid}">
        <button id="{button_id}">
            ▶ Play
        </button>

        <span id="{time_id}">
            0.00 s
        </span>
    </div>

    <script type="module">

        import WaveSurfer from
        "https://cdn.jsdelivr.net/npm/wavesurfer.js@7/dist/wavesurfer.esm.js";

        import RegionsPlugin from
        "https://cdn.jsdelivr.net/npm/wavesurfer.js@7/dist/plugins/regions.esm.js";


        const markers = {markers_json};

        const waveformContainer =
            document.getElementById("{waveform_id}");

        const button =
            document.getElementById("{button_id}");

        const timeDisplay =
            document.getElementById("{time_id}");


        const regions = RegionsPlugin.create();


        const wavesurfer = WaveSurfer.create({{
            container: waveformContainer,

            url: "{audio_url}",

            height: {height},

            waveColor: "#aaa",
            progressColor: "#444",

            cursorColor: "#2563eb",
            cursorWidth: 2,

            normalize: true,

            autoScroll: true,
            autoCenter: true,

            minPxPerSec: 50,

            plugins: [regions],
        }});


        let duration = 0;


        function updateZoom() {{
            const width =
                waveformContainer.clientWidth;

            if (!width) return;

            const pxPerSecond =
                width / {float(window_seconds)};

            wavesurfer.zoom(pxPerSecond);
        }}


        function addMarkers() {{

            regions.clearRegions();

            markers.forEach(marker => {{

                if (
                    marker.time < 0 ||
                    marker.time > duration
                ) {{
                    return;
                }}

                /*
                 * Very narrow region.
                 *
                 * Width is 0.01 sec, which appears
                 * as a vertical red marker when zoomed.
                 */
                const region = regions.addRegion({{
                    start: marker.time,
                    end: Math.min(
                        marker.time + 0.01,
                        duration
                    ),

                    color: "rgba(239, 68, 68, 0.95)",

                    drag: false,
                    resize: false,
                }});


                /*
                 * Tooltip
                 */
                region.element.title =
                    `${{marker.label}} (${{marker.time.toFixed(2)}}s)`;


                /*
                 * Make narrow regions easier to see.
                 */
                region.element.style.minWidth = "2px";
                region.element.style.width = "2px";
                region.element.style.background =
                    "rgb(239, 68, 68)";
                region.element.style.zIndex = "10";
                region.element.style.cursor = "pointer";


                /*
                 * Click marker to seek.
                 */
                region.element.addEventListener(
                    "click",
                    event => {{

                        event.stopPropagation();

                        wavesurfer.setTime(
                            marker.time
                        );
                    }}
                );
            }});
        }}


        wavesurfer.on("ready", () => {{

            duration =
                wavesurfer.getDuration();

            updateZoom();

            requestAnimationFrame(() => {{
                addMarkers();
            }});

            timeDisplay.textContent =
                `0.00 s / ${{duration.toFixed(2)}} s`;
        }});


        wavesurfer.on("timeupdate", time => {{

            timeDisplay.textContent =
                `${{time.toFixed(2)}} s / ` +
                `${{duration.toFixed(2)}} s`;
        }});


        wavesurfer.on("play", () => {{
            button.textContent = "⏸ Pause";
        }});


        wavesurfer.on("pause", () => {{
            button.textContent = "▶ Play";
        }});


        wavesurfer.on("finish", () => {{
            button.textContent = "▶ Play";
        }});


        button.addEventListener("click", () => {{
            wavesurfer.playPause();
        }});


        const resizeObserver =
            new ResizeObserver(() => {{

                if (!duration) return;

                const currentTime =
                    wavesurfer.getCurrentTime();

                updateZoom();

                requestAnimationFrame(() => {{
                    addMarkers();

                    wavesurfer.setTime(
                        currentTime
                    );
                }});
            }});


        resizeObserver.observe(
            waveformContainer
        );

    </script>
    """

    display(HTML(html))

## visualize

In [ ]:
waveform_player(audio, syllable_times, window_seconds=10)

In [ ]:
waveform_player(audio, stress_times, window_seconds=10)

In [ ]:
expected_stresses = [syllable_times[i] for i in [1,3,4,6,7] ]
waveform_player(audio, expected_stresses, window_seconds=10)

In [ ]:
syllable = syllables[1]
waveform_player(audio[:800], [syllable.start, syllable.nucleus, syllable.end])

In [ ]:
waveform_player(audio[:800], stress_times)

In [ ]:
import numpy as np

pitch_prom = np.array([e[0] for e in evidence], dtype=float)
intensity_prom = np.array([e[1] for e in evidence], dtype=float)

pitch_prom = pitch_prom[np.isfinite(pitch_prom)]
intensity_prom = intensity_prom[np.isfinite(intensity_prom)]

pitch_80_threshold = np.percentile(pitch_prom, 20)
intensity_80_threshold = np.percentile(intensity_prom, 20)

print("pitch threshold for ~80% pass:", pitch_80_threshold)
print("intensity threshold for ~80% pass:", intensity_80_threshold)

In [ ]:
print("syllables:", len(syllables))
print("stressed:", len(stresses))
print("stress %:", 100 * len(stresses) / len(syllables))

In [ ]:
for threshold in [0.50, 0.40, 0.30, 0.20, 0.10, 0.00]:
    # flags = [
    #     (score >= threshold and pitch_prom > 0 and intensity_prom > 0)
    #     for (pitch_prom, intensity_prom, _), score in zip(evidence, scores)
    # ]

    flags = classify_stresses(evidence, scores, 0,0,threshold)

    n = sum(flags)
    print(f"{threshold:.2f}: " f"{n}/{len(flags)} " f"({100*n/len(flags):.1f}%)")
    

In [ ]:
pitch_pass = [e[0] >= params.min_pitch_prominence_hz for e in evidence]

intensity_pass = [
    e[1] >= params.min_intensity_prominence_db for e in evidence
]

score_pass = [score >= params.stress_score_threshold for score in scores]

n = len(syllables)

print("pitch:", sum(pitch_pass), f"({100 * sum(pitch_pass) / n:.1f}%)")

print("intensity:", sum(intensity_pass), f"({100 * sum(intensity_pass) / n:.1f}%)")

print(
    "pitch + intensity:",
    sum(p and i for p, i in zip(pitch_pass, intensity_pass)),
    f"({100 * sum(p and i for p, i in zip(pitch_pass, intensity_pass)) / n:.1f}%)",
)

print("score:", sum(score_pass), f"({100 * sum(score_pass) / n:.1f}%)")

print(
    "all:",
    sum(
        p and i and s
        for p, i, s in zip(
            pitch_pass,
            intensity_pass,
            score_pass,
        )
    ),
)